

```
# Isto está formatado como código
```

# Laboratório 6 — Sincronização com Semáforos
**Disciplina:** Programação Concorrente (ICP-361)  
**Profa.:** Silvana Rossetto  
**Instituto de Computação / UFRJ**

---

## Introdução

O objetivo deste laboratório é introduzir e praticar o uso de **semáforos** para implementar **exclusão mútua** e **sincronização condicional** em programas concorrentes.

Neste notebook você irá:
- Compilar e executar programas C com threads diretamente no Colab
- Analisar o comportamento de semáforos em diferentes cenários
- Implementar o padrão **produtor/consumidor** com verificação de primalidade

> 💡 **Como usar este notebook:** execute as células em ordem. Células de código compilam (`gcc`) e executam (`./programa`) os programas C. Células de texto contêm a teoria, perguntas e espaços para suas respostas.

## ⚙️ Configuração do Ambiente

Verifique se o compilador GCC está disponível e instale dependências necessárias.

In [ ]:
# Verifica versão do GCC disponível no ambiente
!gcc --version
!echo "---"
!echo "Ambiente pronto. GCC com suporte a pthreads disponível."

gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

---
Ambiente pronto. GCC com suporte a pthreads disponível.


---
## Atividade 1 — Exclusão Mútua com Semáforos

**Objetivo:** Introduzir o uso de semáforos na linguagem C para implementar exclusão mútua.

### Conceito

Um **semáforo** é uma variável inteira não-negativa com duas operações atômicas:
- `sem_wait(s)` — decrementa o semáforo; se o valor for 0, bloqueia a thread.
- `sem_post(s)` — incrementa o semáforo; se houver threads bloqueadas, desbloqueia uma delas.

Um **semáforo binário** (inicializado com 1) implementa exclusão mútua:

```
sem_wait(&mutex)  →  ENTRADA na seção crítica
   [seção crítica]
sem_post(&mutex)  →  SAÍDA da seção crítica
```

### Programa: `semaf-1.c`

Duas threads incrementam a variável compartilhada `s` 1.000.000 de vezes cada, com exclusão mútua garantida por semáforo. O resultado esperado é `s = 2000000`.

In [ ]:
%%writefile semaf1.c
/* Disciplina: Programacao Concorrente */
/* Prof.: Silvana Rossetto */
/* Comunicacao entre threads com exclusao mutua por semaforo */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <semaphore.h>

#define NTHREADS 2

int s = 0;       // variavel compartilhada
sem_t sem;        // semaforo para exclusao mutua

void *ExecutaTarefa(void *threadid) {
  int i;
  int *tid = (int*) threadid;
  printf("Thread %d executando...\n", *tid);
  for (i = 0; i < 1000000; i++) {
     sem_wait(&sem);   // entrada na secao critica
     s++;             // secao critica
     sem_post(&sem);   // saida da secao critica
  }
  printf("Thread %d terminou!\n", *tid);
  free(threadid);
  pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
  pthread_t tid[NTHREADS];
  int t, *id;

  sem_init(&sem, 0, 1);  // semaforo binario: valor inicial = 1

  for (t = 0; t < NTHREADS; t++) {
    if ((id = malloc(sizeof(int))) == NULL) { pthread_exit(NULL); return 1; }
    *id = t;
    if (pthread_create(&tid[t], NULL, ExecutaTarefa, (void *)id)) {
      printf("ERRO: pthread_create()\n"); exit(-1);
    }
  }
  for (t = 0; t < NTHREADS; t++) {
    if (pthread_join(tid[t], NULL)) {
      printf("ERRO: pthread_join()\n"); exit(-1);
    }
  }
  printf("Valor de s = %d\n", s);
  return 0;
}

Overwriting semaf1.c


In [ ]:
# Compilação
!gcc -o semaf1 semaf1.c -lpthread
print("Compilado com sucesso!")

Compilado com sucesso!


In [ ]:
# Executa 3 vezes para observar consistência
for i in range(5):
    print(f"--- Execução {i+1} ---")
    !./semaf1
    print()

--- Execução 1 ---
Thread 0 executando...
Thread 1 executando...
Thread 1 terminou!
Thread 0 terminou!
Valor de s = 2000000

--- Execução 2 ---
Thread 0 executando...
Thread 1 executando...
Thread 1 terminou!
Thread 0 terminou!
Valor de s = 2000000

--- Execução 3 ---
Thread 1 executando...
Thread 0 executando...
Thread 1 terminou!
Thread 0 terminou!
Valor de s = 2000000

--- Execução 4 ---
Thread 0 executando...
Thread 1 executando...
Thread 0 terminou!
Thread 1 terminou!
Valor de s = 2000000

--- Execução 5 ---
Thread 0 executando...
Thread 1 executando...
Thread 1 terminou!
Thread 0 terminou!
Valor de s = 2000000



### 🔬 Experimento 1.A — Semáforo inicializado com 0

**Antes de executar:** O que vai acontecer se o semáforo for inicializado com **0** sinais? Por quê?

✏️ **Sua resposta:**

> Busy Wait, todas as threads esperam por um sinal para seguir, mas permanecem bloqueadas

In [ ]:
%%writefile semaf1_v0.c
#include <stdio.h>
#include <stdlib.h>
#include <semaphore.h>
#define NTHREADS 2

int s = 0;
sem_t em;

void *ExecutaTarefa(void *threadid) {
  int i, *tid = (int*) threadid;
  printf("Thread %d executando...\n", *tid);
  for (i = 0; i < 1000000; i++) {
     sem_wait(&em);
     s++;
     sem_post(&em);
  }
  printf("Thread %d terminou!\n", *tid);
  free(threadid);
  pthread_exit(NULL);
}

int main() {
  pthread_t tid[NTHREADS];
  int t, *id;
  sem_init(&em, 0, 0);  // <<< ALTERADO: inicializado com 0
  for (t = 0; t < NTHREADS; t++) {
    id = malloc(sizeof(int)); *id = t;
    pthread_create(&tid[t], NULL, ExecutaTarefa, (void *)id);
  }
  for (t = 0; t < NTHREADS; t++) pthread_join(tid[t], NULL);
  printf("Valor de s = %d\n", s);
  return 0;
}

Overwriting semaf1_v0.c


In [ ]:
import subprocess, signal
!gcc -o semaf1_v0 semaf1_v0.c -lpthread
print("Executando com timeout de 4s (deadlock esperado):")
try:
    result = subprocess.run(["./semaf1_v0"], timeout=4, capture_output=True, text=True)
    print(result.stdout)
except subprocess.TimeoutExpired:
    print(">>> TIMEOUT: o programa travou (deadlock) — as threads ficaram bloqueadas em sem_wait!")

semaf1_v0.c: In function ‘ExecutaTarefa’:
semaf1_v0.c:19:3: warning: implicit declaration of function ‘pthread_exit’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   19 |   pthread_exit(NULL);
      |   ^~~~~~~~~~~~
semaf1_v0.c: In function ‘main’:
semaf1_v0.c:28:5: warning: implicit declaration of function ‘pthread_create’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   28 |     pthread_create(&tid[t], NULL, ExecutaTarefa, (void *)id);
      |     ^~~~~~~~~~~~~~
semaf1_v0.c:30:34: warning: implicit declaration of function ‘pthread_join’ []8;;https://gcc.gnu.org/onlinedocs/gcc/Warning-Options.html#index-Wimplicit-function-declaration-Wimplicit-function-declaration]8;;]
   30 |   for (t = 0; t < NTHREADS; t++) pthread_join(tid[t], NULL);
      |                                  ^~~~~~~~~~~~
Executando

✏️ **Análise:** Explique o resultado observado. O que acontece quando `sem_wait` é chamado num semáforo com valor 0?

> _Resposta no formulário._

### 🔬 Experimento 1.B — Semáforo inicializado com 2

**Antes de executar:** O que vai acontecer se o semáforo for inicializado com **2** sinais? Por quê?

✏️ **Sua resposta:**

> _Escreva aqui sua previsão._

In [ ]:
%%writefile semaf1_v2.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <semaphore.h>
#define NTHREADS 2

int s = 0;
sem_t em;

void *ExecutaTarefa(void *threadid) {
  int i, *tid = (int*) threadid;
  printf("Thread %d executando...\n", *tid);
  for (i = 0; i < 100000; i++) {  // reduzido para ser mais rapido
     sem_wait(&em);
     s++;
     sem_post(&em);
  }
  printf("Thread %d terminou!\n", *tid);
  free(threadid);
  pthread_exit(NULL);
}

int main() {
  pthread_t tid[NTHREADS];
  int t, *id;
  sem_init(&em, 0, 2);  // <<< ALTERADO: inicializado com 2
  for (t = 0; t < NTHREADS; t++) {
    id = malloc(sizeof(int)); *id = t;
    pthread_create(&tid[t], NULL, ExecutaTarefa, (void *)id);
  }
  for (t = 0; t < NTHREADS; t++) pthread_join(tid[t], NULL);
  printf("Valor de s = %d (esperado: 200000)\n", s);
  return 0;
}

Overwriting semaf1_v2.c


In [ ]:
!gcc -o semaf1_v2 semaf1_v2.c -lpthread
print("Executando 3 vezes:")
for i in range(3):
    print(f"Execução {i+1}:", end=" ")
    !./semaf1_v2 | tail -1

Executando 3 vezes:
Execução 1: Valor de s = 199673 (esperado: 200000)
Execução 2: Valor de s = 199195 (esperado: 200000)
Execução 3: Valor de s = 165620 (esperado: 200000)


✏️ **Análise:** O resultado foi o esperado (200000)? Por que inicializar com 2 quebra a exclusão mútua?

> _Sua resposta aqui._

---
## Atividade 2 — Ordem de Execução com Semáforos

**Objetivo:** Usar semáforos para impor uma ordem de execução específica entre threads.

### Conceito

Um semáforo inicializado com **0** pode ser usado como **condição de sincronização**:
- A thread que *depende* de outra faz `sem_wait` (bloqueia até ser sinalizada).
- A thread que *libera* a dependência faz `sem_post`.

### Programa: `semaf-2.c`

Três threads devem executar em ordem: **T1 → T2 → T3**.

| Thread | Ação | Dependência |
|--------|------|-------------|
| T1 | seta `x = 1`; sinaliza T2 | nenhuma |
| T2 | espera T1; seta `x = 2`; sinaliza T3 | espera `condt2` |
| T3 | espera T2; imprime `x` | espera `condt3` |

**Resultado esperado:** `x = 2` (T2 foi a última a escrever antes de T3 ler)

In [ ]:
%%writefile semaf2.c
/* Disciplina: Programacao Concorrente */
/* Prof.: Silvana Rossetto */
/* Ordem de execucao com semaforos */

#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <semaphore.h>

#define NTHREADS 3

int x = 0;
sem_t condt2, condt3;  // semaforos de sincronizacao condicional

void *t1(void *threadid) {
  int *tid = (int*) threadid;
  printf("Thread %d executando...\n", *tid);
  x = 1;
  sem_post(&condt2);  // permite que T2 execute
  printf("Thread %d terminou!\n", *tid);
  pthread_exit(NULL);
}

void *t2(void *threadid) {
  int *tid = (int*) threadid;
  printf("Thread %d executando...\n", *tid);
  sem_wait(&condt2);  // espera T1 executar
  x = 2;
  sem_post(&condt3);  // permite que T3 execute
  printf("Thread %d terminou!\n", *tid);
  pthread_exit(NULL);
}

void *t3(void *threadid) {
  int *tid = (int*) threadid;
  printf("Thread %d executando...\n", *tid);
  sem_wait(&condt3);  // espera T2 executar
  printf("Valor de x = %d\n", x);
  printf("Thread %d terminou!\n", *tid);
  pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
  pthread_t tid[NTHREADS];
  int *id[3], t;

  for (t = 0; t < NTHREADS; t++) {
    id[t] = malloc(sizeof(int));
    *id[t] = t + 1;
  }

  sem_init(&condt2, 0, 0);  // inicializado com 0 para bloquear T2
  sem_init(&condt3, 0, 0);  // inicializado com 0 para bloquear T3

  // As threads sao criadas em ordem diferente da execucao!
  pthread_create(&tid[1], NULL, t2, (void *)id[1]);
  pthread_create(&tid[2], NULL, t3, (void *)id[2]);
  pthread_create(&tid[0], NULL, t1, (void *)id[0]);

  for (t = 0; t < NTHREADS; t++) {
    pthread_join(tid[t], NULL);
    free(id[t]);
  }
  return 0;
}

Overwriting semaf2.c


In [ ]:
!gcc -o semaf2 semaf2.c -lpthread
print("Executando 4 vezes para observar consistência da ordem:")
for i in range(4):
    print(f"--- Execução {i+1} ---")
    !./semaf2
    print()

Executando 4 vezes para observar consistência da ordem:
--- Execução 1 ---
Thread 3 executando...
Thread 2 executando...
Thread 1 executando...
Thread 2 terminou!
Valor de x = 2
Thread 3 terminou!
Thread 1 terminou!

--- Execução 2 ---
Thread 3 executando...
Thread 2 executando...
Thread 1 executando...
Thread 1 terminou!
Thread 2 terminou!
Valor de x = 2
Thread 3 terminou!

--- Execução 3 ---
Thread 2 executando...
Thread 3 executando...
Thread 1 executando...
Thread 1 terminou!
Thread 2 terminou!
Valor de x = 2
Thread 3 terminou!

--- Execução 4 ---
Thread 3 executando...
Thread 2 executando...
Thread 1 executando...
Thread 1 terminou!
Thread 2 terminou!
Valor de x = 2
Thread 3 terminou!



✏️ **Questão 2.1:** Por que os semáforos `condt2` e `condt3` devem ser inicializados com **0**? O que aconteceria se fossem inicializados com **1**?

> _Resposta no formulário._

### 🔬 Experimento 2 — Semáforos inicializados com 1

In [ ]:
%%writefile semaf2_v1.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <semaphore.h>
#define NTHREADS 3

int x = 0;
sem_t condt2, condt3;

void *t1(void *threadid) {
  int *tid = (int*) threadid;
  x = 1; sem_post(&condt2);
  printf("T%d: x=%d (escreveu 1)\n", *tid, x);
  pthread_exit(NULL);
}
void *t2(void *threadid) {
  int *tid = (int*) threadid;
  sem_wait(&condt2); x = 2; sem_post(&condt3);
  printf("T%d: x=%d (escreveu 2)\n", *tid, x);
  pthread_exit(NULL);
}
void *t3(void *threadid) {
  int *tid = (int*) threadid;
  sem_wait(&condt3);
  printf("T%d: x = %d (leu)\n", *tid, x);
  pthread_exit(NULL);
}

int main() {
  pthread_t tid[NTHREADS];
  int *id[3], t;
  for (t = 0; t < NTHREADS; t++) { id[t] = malloc(sizeof(int)); *id[t] = t+1; }
  sem_init(&condt2, 0, 1);  // <<< ALTERADO: inicializado com 1
  sem_init(&condt3, 0, 1);  // <<< ALTERADO: inicializado com 1
  pthread_create(&tid[1], NULL, t2, (void *)id[1]);
  pthread_create(&tid[2], NULL, t3, (void *)id[2]);
  pthread_create(&tid[0], NULL, t1, (void *)id[0]);
  for (t = 0; t < NTHREADS; t++) { pthread_join(tid[t], NULL); free(id[t]); }
  return 0;
}

Overwriting semaf2_v1.c


In [ ]:
!gcc -o semaf2_v1 semaf2_v1.c -lpthread
print("Com semáforos = 1 (ordem quebrada):")
for i in range(4):
    print(f"--- Execução {i+1} ---")
    !./semaf2_v1
    print()

Com semáforos = 1 (ordem quebrada):
--- Execução 1 ---
T2: x=2 (escreveu 2)
T3: x = 2 (leu)
T1: x=1 (escreveu 1)

--- Execução 2 ---
T2: x=2 (escreveu 2)
T3: x = 2 (leu)
T1: x=1 (escreveu 1)

--- Execução 3 ---
T2: x=2 (escreveu 2)
T3: x = 2 (leu)
T1: x=1 (escreveu 1)

--- Execução 4 ---
T2: x=2 (escreveu 2)
T3: x = 2 (leu)
T1: x=1 (escreveu 1)



✏️ **Análise:** O valor de `x` impresso por T3 foi sempre 2? O que mudou com semáforos inicializados com 1? Explique.

> _Resposta no formulário._

---
## Atividade 3 — Barreira com Semáforos

**Objetivo:** Analisar uma implementação do padrão **barreira** e identificar versões corretas e incorretas.

### Conceito

Uma **barreira** é um ponto de sincronização onde todas as threads devem chegar antes que qualquer uma possa continuar. É útil em computação iterativa.

```
Thread 1: ===passo1===|   (espera)   |===passo2===|...
Thread 2: ==passo1==  |   (espera)   |===passo2===|...
Thread 3: ====passo1==|   (espera)   |===passo2===|...
                      ↑ BARREIRA     ↑ BARREIRA
```

### Programa: `barreira.c`

5 threads executam 4 passos. Em cada passo: incrementam um contador, imprimem, e esperam na barreira.

In [ ]:
%%writefile barreira.c
/* Disciplina: Computacao Concorrente */
/* Profa.: Silvana Rossetto */
/* Barreira usando semaforos */

#include <pthread.h>
#include <stdio.h>
#include <stdlib.h>
#include <semaphore.h>

#define NTHREADS  5
#define PASSOS    4

int bloqueadas = 0;
sem_t mutex, cond;

/* Barreira incorreta (versao 0) */
void barreira0(int numThreads) {
    int i;
    sem_wait(&mutex);
    bloqueadas++;
    if (bloqueadas < numThreads) {
      sem_post(&mutex);
      sem_wait(&cond);
    } else {
      printf("\n");
      for(i=0; i<(numThreads-1); i++) { sem_post(&cond); }
      bloqueadas = 0;
      sem_post(&mutex);
    }
}

/* Barreira correta */
void barreira(int numThreads) {
    sem_wait(&mutex);
    bloqueadas++;
    if (bloqueadas < numThreads) {
      sem_post(&mutex);
      sem_wait(&cond);
      bloqueadas--;
      if (bloqueadas == 0) sem_post(&mutex);
      else sem_post(&cond);
    } else {
      printf("\n");
      bloqueadas--;
      sem_post(&cond);
    }
}

/* Barreira incorreta (versao 1) */
void barreira1(int numThreads) {
    sem_wait(&mutex);
    bloqueadas++;
    if (bloqueadas < numThreads) {
      sem_post(&mutex);
      sem_wait(&cond);
      bloqueadas--;
      if (bloqueadas == 0) sem_post(&mutex);
      else sem_post(&cond);
    } else {
      printf("\n");
      sem_post(&cond);
      bloqueadas--;   // BUG: decrementa fora do mutex
    }
}

void *A(void *t) {
  int my_id = *(int*)t, i;
  int boba1, boba2;
  for (i = 0; i < PASSOS; i++) {
    printf("Thread %d: passo=%d\n", my_id, i);
    barreira(NTHREADS);  // use barreira() - versao correta
    boba1=100; boba2=-100; while(boba2 < boba1) boba2++;
  }
  pthread_exit(NULL);
}

int main(int argc, char *argv[]) {
  int i;
  pthread_t threads[NTHREADS];
  int id[NTHREADS];

  sem_init(&mutex, 0, 1);   // Valor inicial do semáforo mutex
  sem_init(&cond,  0, 0);   // Valor inicial do semáforo cond

  for(i = 0; i < NTHREADS; i++) { id[i]=i; pthread_create(&threads[i], NULL, A, &id[i]); }
  for(i = 0; i < NTHREADS; i++) { pthread_join(threads[i], NULL); }

  sem_destroy(&mutex);
  sem_destroy(&cond);
  printf("FIM.\n");
  return 0;
}

Overwriting barreira.c


In [ ]:
!gcc -o barreira barreira.c -lpthread
print("Executando (versão correta da barreira):")
print("Observe que todos os passos de todas as threads aparecem antes de cada separação.")
print()
!./barreira

Executando (versão correta da barreira):
Observe que todos os passos de todas as threads aparecem antes de cada separação.

Thread 0: passo=0
Thread 2: passo=0
Thread 3: passo=0
Thread 4: passo=0
Thread 1: passo=0

Thread 1: passo=1
Thread 0: passo=1
Thread 3: passo=1
Thread 2: passo=1
Thread 4: passo=1

Thread 0: passo=2
Thread 4: passo=2
Thread 2: passo=2
Thread 3: passo=2
Thread 1: passo=2

Thread 0: passo=3
Thread 2: passo=3
Thread 3: passo=3
Thread 1: passo=3
Thread 4: passo=3

FIM.


### Análise das versões incorretas

Veja abaixo o comportamento esperado de cada versão:

| Versão | Correta? | Problema |
|--------|----------|----------|
| `barreira0` | ❌ | Após liberar threads, `mutex` fica liberado (sem_post) mas as threads que acordaram podem competir com a próxima rodada antes de `bloqueadas` ser resetado |
| `barreira` | ✅ | Usa cascade: cada thread acorda a próxima e a última libera o `mutex` |
| `barreira1` | ❌ | `bloqueadas--` ocorre *fora* do `mutex`, gerando condição de corrida |

✏️ **Questão 3.1:** Explique detalhadamente por que `barreira0` está incorreta. Descreva um cenário de execução que demonstre o problema.

> _Resposta no formulário._

✏️ **Questão 3.2:** Por que `barreira1` está incorreta? Que condição de corrida pode ocorrer?

> _Resposta no formulário._

---
## Atividade 4 — Produtor/Consumidor com Semáforos

**Objetivo:** Analisar a implementação do padrão **produtor/consumidor** com semáforos.

### Conceito

O padrão produtor/consumidor usa **três semáforos**:

| Semáforo | Valor inicial | Propósito |
|----------|---------------|-----------|
| `slotVazio` | N (tamanho do buffer) | Conta slots livres |
| `slotCheio` | 0 | Conta slots ocupados |
| `mutexGeral` | 1 | Exclusão mútua no buffer |

```
PRODUTOR:                    CONSUMIDOR:
sem_wait(slotVazio)          sem_wait(slotCheio)
sem_wait(mutex)              sem_wait(mutex)
  [insere no buffer]           [retira do buffer]
sem_post(mutex)              sem_post(mutex)
sem_post(slotCheio)          sem_post(slotVazio)
```

### Programa: `pc.c`

10 produtores e 5 consumidores compartilham um buffer circular de tamanho 5.

In [ ]:
%%writefile pc.c
/* Disciplina: Programacao Concorrente */
/* Prof.: Silvana Rossetto */
/* Produtor/consumidor usando semaforos */

#include <pthread.h>
#include <stdlib.h>
#include <stdio.h>
#include <semaphore.h>
#include <unistd.h>

#define PRODUTORES  10
#define CONSUMIDORES 5
#define MAX (PRODUTORES+CONSUMIDORES)
#define N   5  // tamanho do buffer

sem_t slotCheio, slotVazio, mutexGeral;
int Buffer[N];

void printBuffer(int buffer[], int tam) {
   for(int i=0;i<tam;i++) printf("%d ", buffer[i]);
   puts("");
}

void Insere(int item, int id) {
   static int in=0;
   sem_wait(&slotVazio);
   sem_wait(&mutexGeral);
   Buffer[in] = item;
   in = (in + 1) % N;
   printf("Prod[%d]: inseriu %d | Buffer: ", id, item);
   printBuffer(Buffer, N);
   sem_post(&mutexGeral);
   sem_post(&slotCheio);
}

int Retira(int id) {
   int item;
   static int out=0;
   sem_wait(&slotCheio);
   sem_wait(&mutexGeral);
   item = Buffer[out];
   Buffer[out] = 0;
   out = (out + 1) % N;
   printf("Cons[%d]: retirou %d | Buffer: ", id, item);
   printBuffer(Buffer, N);
   sem_post(&mutexGeral);
   sem_post(&slotVazio);
   return item;
}

void *produtor(void *arg) {
  int id = *(int *)(arg); free(arg);
  while(1) { sleep(1); Insere(id, id); }
  pthread_exit(NULL);
}

void *consumidor(void *arg) {
  int item, id = *(int *)(arg); free(arg);
  while(1) { item = Retira(id); sleep(1); }
  pthread_exit(NULL);
}

int main(int argc, char **argv) {
  pthread_t tid[MAX];
  int i, *id;

  sem_init(&mutexGeral, 0, 1);
  sem_init(&slotCheio,  0, 0);
  sem_init(&slotVazio,  0, N);

  for(i=0;i<PRODUTORES;i++) {
    id = malloc(sizeof(int)); *id = i+1;
    pthread_create(&tid[i], NULL, produtor, id);
  }
  for(i=0;i<CONSUMIDORES;i++) {
    id = malloc(sizeof(int)); *id = i+1;
    pthread_create(&tid[PRODUTORES+i], NULL, consumidor, id);
  }
  pthread_exit(NULL);
}

Overwriting pc.c


In [ ]:
import subprocess
!gcc -o pc pc.c -lpthread
print("Executando por 5 segundos (loop infinito, interrompido automaticamente):")
print()
try:
    result = subprocess.run(["./pc"], timeout=10, capture_output=True, text=True)
    print(result.stdout[:3000])
except subprocess.TimeoutExpired as e:
    output = e.stdout.decode() if e.stdout else ""
    print(output[:3000])
    print("\n[Programa interrompido após 5s — execução contínua esperada]")

Executando por 5 segundos (loop infinito, interrompido automaticamente):



[Programa interrompido após 5s — execução contínua esperada]


✏️ **Questão 4.1:** Qual é a finalidade das variáveis `in` e `out`? Por que foram declaradas como `static` dentro das funções?

> _Resposta no formulário._

✏️ **Questão 4.2:** Como verificar que o programa executou corretamente? Que invariante deve ser mantida?

> _Resposta no formulário._

---
## Atividade 5 — Exemplo de Uso de Semáforos

## Funcionamento

O programa modela um **estacionamento com 3 vagas** e 6 carros disputando acesso simultâneo. O semáforo aqui não é binário (0/1), mas um **semáforo contador** inicializado com `VAGAS = 3` — ele representa quantas vagas ainda estão disponíveis.

Cada carro (thread) executa três fases: tenta entrar → `sem_wait` decrementa o semáforo (ou bloqueia se = 0) → usa a vaga por 2s → `sem_post` libera a vaga e acorda o próximo carro esperando. Nunca mais de 3 threads estão dentro da seção crítica ao mesmo tempo.**Invariante garantida pelo semáforo:** o valor do contador nunca fica negativo — representa exatamente quantas vagas livres existem a qualquer momento. Com valor 0, todas as chamadas a `sem_wait` bloqueiam até que alguma thread execute `sem_post`.

## Atividades Didáticas

**Atividade 1 — Observar o comportamento base**

Execute o código original várias vezes e anote: os carros 1–3 sempre entram antes de 4–6? A ordem de desbloqueio da fila é sempre a mesma? Discuta o que determina essa ordem?

**Atividade 2 — Variar parâmetros**

Altere `VAGAS` e `NUM_THREADS` para explorar os casos extremos:
- `VAGAS = NUM_THREADS`
- `VAGAS = 1`
- `VAGAS = 0`

**Atividade 3 — Instrumentar o semáforo**

Adicione uma variável `vagas_livres` que é decrementada/incrementada junto com `sem_wait`/`sem_post` (protegida por um mutex separado) e imprima seu valor em cada evento. Verifique os valores máximo e mínimo que assume.

**Atividade 4 — Simular tempo de estadia variável**

Substitua `sleep(2)` por `sleep(rand() % 4 + 1)` para tempos aleatórios entre 1 e 4s. Observe como a dinâmica da fila muda. Algum carro pode esperar indefinidamente?

**Atividade 5 — Semáforo vs. mutex vs. variável de condição**

Reimplemente o mesmo problema usando `pthread_mutex_t` + `pthread_cond_t` (como no arquivo `prod_consum.c` do laboratório). Compare as duas soluções em termos de legibilidade, quantidade de código e facilidade de estender para múltiplos tipos de recurso.

##_Respostas no formulário._


In [5]:
%%writefile vagas_sem.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <semaphore.h>
#include <unistd.h>

#define NUM_THREADS 6
#define VAGAS 3

sem_t semaforo;

int vagas_livres = VAGAS;
pthread_mutex_t mutex;

void* carro(void *arg) {
    int id = *(int*)arg;

    printf("Carro %d: tentando entrar...\n", id);

    sem_wait(&semaforo);

    pthread_mutex_lock(&mutex);
    vagas_livres--;
    printf("Carro %d: entrou na vaga. Vagas livres = %d\n",
           id, vagas_livres);
    pthread_mutex_unlock(&mutex);

    sleep(rand() % 4 + 1);

    pthread_mutex_lock(&mutex);
    vagas_livres++;
    printf("Carro %d: saiu da vaga. Vagas livres = %d\n",
           id, vagas_livres);
    pthread_mutex_unlock(&mutex);

    sem_post(&semaforo);

    return NULL;
}

int main() {
    pthread_t threads[NUM_THREADS];
    int ids[NUM_THREADS];

    sem_init(&semaforo, 0, VAGAS);
    pthread_mutex_init(&mutex, NULL);

    for (int i = 0; i < NUM_THREADS; i++) {
        ids[i] = i + 1;
        pthread_create(&threads[i], NULL, carro, &ids[i]);
    }

    for (int i = 0; i < NUM_THREADS; i++) {
        pthread_join(threads[i], NULL);
    }

    sem_destroy(&semaforo);
    pthread_mutex_destroy(&mutex);

    return 0;
}

Overwriting vagas_sem.c


In [7]:
!gcc -o teste vagas_sem.c -lpthread
!./teste

Carro 1: tentando entrar...
Carro 1: entrou na vaga. Vagas livres = 2
Carro 2: tentando entrar...
Carro 2: entrou na vaga. Vagas livres = 1
Carro 3: tentando entrar...
Carro 3: entrou na vaga. Vagas livres = 0
Carro 4: tentando entrar...
Carro 5: tentando entrar...
Carro 6: tentando entrar...
Carro 3: saiu da vaga. Vagas livres = 1
Carro 4: entrou na vaga. Vagas livres = 0
Carro 2: saiu da vaga. Vagas livres = 1
Carro 5: entrou na vaga. Vagas livres = 0
Carro 1: saiu da vaga. Vagas livres = 1
Carro 6: entrou na vaga. Vagas livres = 0
Carro 5: saiu da vaga. Vagas livres = 1
Carro 4: saiu da vaga. Vagas livres = 2
Carro 6: saiu da vaga. Vagas livres = 3


In [8]:
%%writefile vagas_sem.c
#include <stdio.h>
#include <stdlib.h>
#include <pthread.h>
#include <unistd.h>
#include <time.h>

#define NUM_THREADS 6
#define VAGAS_TOTAIS 3

// Estado do monitor
int vagas_livres = VAGAS_TOTAIS;
pthread_mutex_t mutex;
pthread_cond_t cond_vaga_disponivel;

void* carro(void *arg) {
    int id = *(int*)arg;

    printf("Carro %d: chegando ao estacionamento...\n", id);

    // --- ENTRADA (PROTOCÓLO DE ACESSO) ---
    pthread_mutex_lock(&mutex);

    // IMPORTANTE: O 'while' é obrigatório para evitar acordares espúrios
    while (vagas_livres == 0) {
        printf("Carro %d: estacionamento lotado. Aguardando vaga...\n", id);
        pthread_cond_wait(&cond_vaga_disponivel, &mutex);
    }

    vagas_livres--;
    printf("Carro %d: estacionou! Vagas restantes = %d\n", id, vagas_livres);

    pthread_mutex_unlock(&mutex);

    // --- SEÇÃO CRÍTICA (ESTACIONADO) ---
    sleep(rand() % 4 + 1);

    // --- SAÍDA (PROTOCÓLO DE LIBERAÇÃO) ---
    pthread_mutex_lock(&mutex);

    vagas_livres++;
    printf("Carro %d: saiu. Vagas agora livres = %d\n", id, vagas_livres);

    // Notifica que uma nova vaga surgiu
    pthread_cond_signal(&cond_vaga_disponivel);

    pthread_mutex_unlock(&mutex);

    return NULL;
}

int main() {
    pthread_t threads[NUM_THREADS];
    int ids[NUM_THREADS];

    srand(time(NULL));
    pthread_mutex_init(&mutex, NULL);
    pthread_cond_init(&cond_vaga_disponivel, NULL);

    for (int i = 0; i < NUM_THREADS; i++) {
        ids[i] = i + 1;
        pthread_create(&threads[i], NULL, carro, &ids[i]);
    }

    for (int i = 0; i < NUM_THREADS; i++) {
        pthread_join(threads[i], NULL);
    }

    pthread_mutex_destroy(&mutex);
    pthread_cond_destroy(&cond_vaga_disponivel);

    return 0;
}

Overwriting vagas_sem.c


In [9]:
!gcc -o teste vagas_sem.c -lpthread
!./teste

Carro 1: chegando ao estacionamento...
Carro 1: estacionou! Vagas restantes = 2
Carro 2: chegando ao estacionamento...
Carro 2: estacionou! Vagas restantes = 1
Carro 3: chegando ao estacionamento...
Carro 3: estacionou! Vagas restantes = 0
Carro 4: chegando ao estacionamento...
Carro 4: estacionamento lotado. Aguardando vaga...
Carro 5: chegando ao estacionamento...
Carro 5: estacionamento lotado. Aguardando vaga...
Carro 6: chegando ao estacionamento...
Carro 6: estacionamento lotado. Aguardando vaga...
Carro 1: saiu. Vagas agora livres = 1
Carro 4: estacionou! Vagas restantes = 0
Carro 2: saiu. Vagas agora livres = 1
Carro 5: estacionou! Vagas restantes = 0
Carro 3: saiu. Vagas agora livres = 1
Carro 6: estacionou! Vagas restantes = 0
Carro 4: saiu. Vagas agora livres = 1
Carro 5: saiu. Vagas agora livres = 2
Carro 6: saiu. Vagas agora livres = 3


---
## Atividade 6 — Implementação: Produtor/Consumidor com Primalidade

**Objetivo:** Projetar e implementar um programa concorrente em C usando o padrão produtor/consumidor.

### Descrição do Problema

Implemente um programa concorrente onde:

- **1 thread PRODUTORA** gera uma sequência de **N números inteiros** e os deposita num **buffer circular de tamanho M** (um de cada vez).
- **1 thread CONSUMIDORA** retira os números (um de cada vez) e avalia a **primalidade** de cada um.
- **N** e **M** são lidos da entrada padrão.
- Ao final, deve ser impresso o **total de primos encontrados**.

### Função de primalidade

```c
int ehPrimo(long long int n) {
    int i;
    if (n <= 1) return 0;
    if (n == 2) return 1;
    if (n % 2 == 0) return 0;
    for (i = 3; i < sqrt(n) + 1; i += 2)
        if (n % i == 0) return 0;
    return 1;
}
```

### Protocolo de Término

A thread consumidora precisa saber quando parou de receber dados. Uma estratégia comum é o **token de fim**: o produtor insere um valor sentinela (ex: `-1`) como último elemento, sinalizando ao consumidor que a produção acabou.

### Implementação

Implemente o seu código a seguir.

In [ ]:
# Compilação com -lm para a função sqrt()
!gcc -o primo_pc primo_pc.c -lpthread -lm
print("Compilado com sucesso!")

cc1: fatal error: primo_pc.c: No such file or directory
compilation terminated.
Compilado com sucesso!


In [ ]:
# Teste 1: N=50, M=5
print("=== Teste 1: N=50, M=5 ===")
print("(Esperado: 15 primos em [1,50])")
!echo '50 5' | ./primo_pc
print()

# Teste 2: N=100, M=3
print("=== Teste 2: N=100, M=3 ===")
print("(Esperado: 25 primos em [1,100])")
!echo '100 3' | ./primo_pc
print()

# Teste 3: N=1000, M=10
print("=== Teste 3: N=1000, M=10 ===")
print("(Esperado: 168 primos em [1,1000])")
!echo '1000 10' | ./primo_pc

=== Teste 1: N=50, M=5 ===
(Esperado: 15 primos em [1,50])
/bin/bash: line 1: ./primo_pc: No such file or directory

=== Teste 2: N=100, M=3 ===
(Esperado: 25 primos em [1,100])
/bin/bash: line 1: ./primo_pc: No such file or directory

=== Teste 3: N=1000, M=10 ===
(Esperado: 168 primos em [1,1000])
/bin/bash: line 1: ./primo_pc: No such file or directory


In [ ]:
# Verificação sequencial (ground truth)
def eh_primo(n):
    if n <= 1: return False
    if n == 2: return True
    if n % 2 == 0: return False
    import math
    for i in range(3, int(math.sqrt(n)) + 1, 2):
        if n % i == 0: return False
    return True

for N in [50, 100, 1000]:
    count = sum(1 for i in range(1, N+1) if eh_primo(i))
    print(f"Primos em [1,{N:4d}] = {count}  (verificação Python sequencial)")

Primos em [1,  50] = 15  (verificação Python sequencial)
Primos em [1, 100] = 25  (verificação Python sequencial)
Primos em [1,1000] = 168  (verificação Python sequencial)


✏️ **Questão 6.1:** Os resultados do programa concorrente batem com a verificação sequencial? O que isso nos diz sobre a corretude da sincronização?

> _Resposta no formulário.._

✏️ **Questão 6.2:** Por que é necessário um **token sentinela** (-1) para terminar o programa? Existe outra abordagem possível?

> _Resposta no formulário._

### 🔬 Desafio: Múltiplos Consumidores

Modifique o programa para ter **C consumidores** (lido da entrada). Cada consumidor processa parte dos números. O total de primos deve ser a soma de todos os consumidores.

In [ ]:
!gcc -o primo_pc_multi primo_pc_multi.c -lpthread -lm

print("=== Multi-consumidor: N=1000, M=5, C=3 ===")
print("(Esperado: 168 primos em [1,1000])")
!echo '1000 5 3' | ./primo_pc_multi
print()

print("=== Multi-consumidor: N=1000, M=10, C=5 ===")
!echo '1000 10 5' | ./primo_pc_multi

cc1: fatal error: primo_pc_multi.c: No such file or directory
compilation terminated.
=== Multi-consumidor: N=1000, M=5, C=3 ===
(Esperado: 168 primos em [1,1000])
/bin/bash: line 1: ./primo_pc_multi: No such file or directory

=== Multi-consumidor: N=1000, M=10, C=5 ===
/bin/bash: line 1: ./primo_pc_multi: No such file or directory


✏️ **Questão Desafio:** No programa com múltiplos consumidores, por que são necessárias **C sentinelas** (-1) em vez de apenas uma? O que aconteceria com apenas uma sentinela e vários consumidores?

> _Resposta no formulário._

---
## 📋 Resumo do Laboratório

| Atividade | Conceito | Mecanismo |
|-----------|----------|-----------|
| 1 | Exclusão mútua | Semáforo binário (`init=1`) |
| 2 | Ordem de execução | Semáforo condicional (`init=0`) |
| 3 | Barreira | Combinação de mutex e cond com cascade |
| 4 | Produtor/Consumidor | 3 semáforos: slotVazio, slotCheio, mutex |
| 5 | Semáforos | 3 vagas, 6 carros |
| 6 | Implementação completa | Produtor/Consumidor + primalidade |

### Regras de ouro para semáforos

1. **Exclusão mútua** → inicialize com **1**
2. **Sincronização condicional** → inicialize com **0**
3. **Nunca** esqueça o `sem_post` correspondente a um `sem_wait`
4. **Não aninhe** semáforos de exclusão mútua sem cuidado (risco de deadlock)
5. Em produtor/consumidor, o **mutex** deve estar **dentro** das operações de slot (não fora)